In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# 1. LOAD FLOOD DATASET
# ==========================================

df = pd.read_excel("Final_INDOFLOODS_dataset(4).xlsx")

# ==========================================
# 2. CLEAN REQUIRED COLUMNS
# ==========================================

df["population_within_5km"] = pd.to_numeric(
    df["population_within_5km"], errors="coerce"
).fillna(0)

df["Event Duration (days)"] = pd.to_numeric(
    df["Event Duration (days)"], errors="coerce"
).fillna(1)

# Minimum 1 day
df["Event Duration (days)"] = df["Event Duration (days)"].clip(lower=1)

# ==========================================
# 3. GET MODEL PREDICTION
# ==========================================

# If your Random Forest model predicts an impact score:
# df["Impact_Score"] = model.predict(X)

# ------------------------------------------------
# TEMPORARY DEMO ONLY:
# Replace this with your actual model prediction.
# ------------------------------------------------

df["Impact_Score"] = 50

# ==========================================
# 4. CONVERT IMPACT SCORE TO MULTIPLIER
# ==========================================

def get_multiplier(score):
    if score <= 20:
        return 1.0
    elif score <= 40:
        return 1.25
    elif score <= 60:
        return 1.5
    elif score <= 80:
        return 1.75
    else:
        return 2.0

df["Severity_Multiplier"] = df["Impact_Score"].apply(
    get_multiplier
)

# ==========================================
# 5. RESOURCE REQUIREMENT FORMULAS
# ==========================================

people = df["population_within_5km"]
days = df["Event Duration (days)"]
m = df["Severity_Multiplier"]

# Water: 5 litres per person per day
# Assuming 1 bottle = 1 litre
df["Water_Bottles_Required"] = np.ceil(
    5 * people * days * m
).astype(int)

# Food: 3 packets per person per day
df["Food_Packets_Required"] = np.ceil(
    3 * people * days * m
).astype(int)

# Medkits: 1 per 20 people, adjusted by severity
df["Medkits_Required"] = np.ceil(
    (people / 20) * m
).astype(int)

# Blankets: 70% of affected population estimated to need blankets
df["Blankets_Required"] = np.ceil(
    people * 0.70 * m
).astype(int)

# ==========================================
# 6. CREATE RESOURCE TABLE
# ==========================================

resource_table = df[
    [
        "EventID",
        "Station",
        "State",
        "Latitude",
        "Longitude",
        "population_within_5km",
        "Event Duration (days)",
        "Impact_Score",
        "Severity_Multiplier",
        "Water_Bottles_Required",
        "Food_Packets_Required",
        "Medkits_Required",
        "Blankets_Required"
    ]
]

# ==========================================
# 7. SAVE OUTPUT
# ==========================================

resource_table.to_csv(
    "flood_resource_requirements.csv",
    index=False
)

resource_table.to_excel(
    "flood_resource_requirements.xlsx",
    index=False
)

print(resource_table.head())
print("\nResource requirement table created successfully!")